# Week 15 V5: CodeGnosis-Triggered Retrieval

## The Full Pipeline

1. **Generate answer** and extract CodeGnosis features (hidden states + attention + CCE)
2. **Predict if answer is wrong** using trained classifier
3. **If predicted wrong** → trigger retrieval and regenerate
4. **Measure**: Did retrieval fix the predicted errors?

## Key Metrics
- **Error Prediction Accuracy**: How well does CodeGnosis predict errors?
- **Retrieval Fix Rate**: When we retrieve, how often does it fix the error?
- **Overall Improvement**: Final accuracy vs baseline

In [1]:
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers scikit-learn

In [2]:
import os
import json
import torch
import torch.nn.functional as F
import numpy as np
from typing import List, Dict, Tuple
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [3]:
# Clone httpx
!git clone --depth 1 https://github.com/encode/httpx.git /content/httpx 2>/dev/null || true

def load_source_files(repo_dir: str) -> Dict[str, str]:
    files = {}
    for root, _, filenames in os.walk(os.path.join(repo_dir, 'httpx')):
        for fname in filenames:
            if fname.endswith('.py'):
                fpath = os.path.join(root, fname)
                rel_path = os.path.relpath(fpath, repo_dir)
                with open(fpath, 'r', encoding='utf-8') as f:
                    files[rel_path] = f.read()
    return files

SOURCE_FILES = load_source_files('/content/httpx')
print(f"Loaded {len(SOURCE_FILES)} source files")

Loaded 23 source files


In [4]:
# Load model with output_hidden_states for CodeGnosis
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    output_hidden_states=True,  # For CodeGnosis
    output_attentions=True,     # For CodeGnosis
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Models loaded!")

The following generation flags are not valid and may be ignored: ['output_attentions', 'output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded!


## Tasks with Ground Truth for Verification

In [5]:
# Tasks with verification functions
TASKS = [
    {
        'id': 1,
        'question': 'What are ALL the parameters of httpx.Timeout.__init__()? List each parameter name.',
        'relevant_file': 'httpx/_config.py',
        'verify': lambda r: all(p in r.lower() for p in ['timeout', 'connect', 'read', 'write', 'pool']) and 'total' not in r.lower(),
        'ground_truth': 'timeout, connect, read, write, pool',
    },
    {
        'id': 2,
        'question': 'What is the default timeout value in httpx? Give the exact number in seconds.',
        'relevant_file': 'httpx/_config.py',
        'verify': lambda r: '5' in r and '30' not in r,
        'ground_truth': '5.0 seconds',
    },
    {
        'id': 3,
        'question': 'What are the parameters of httpx.Limits.__init__()? List each parameter name.',
        'relevant_file': 'httpx/_config.py',
        'verify': lambda r: 'max_connections' in r.lower() and 'keepalive_expiry' in r.lower() and 'max_retries' not in r.lower(),
        'ground_truth': 'max_connections, max_keepalive_connections, keepalive_expiry',
    },
    {
        'id': 4,
        'question': 'What is the default value for max_connections in httpx.Limits?',
        'relevant_file': 'httpx/_config.py',
        'verify': lambda r: 'none' in r.lower() and '100' not in r,
        'ground_truth': 'None',
    },
    {
        'id': 5,
        'question': 'What is the parent class of ConnectError in httpx._exceptions?',
        'relevant_file': 'httpx/_exceptions.py',
        'verify': lambda r: 'networkerror' in r.lower() and 'exception' not in r.lower().replace('networkerror', ''),
        'ground_truth': 'NetworkError',
    },
    {
        'id': 6,
        'question': 'What exception does httpx raise when the URL scheme is not http or https?',
        'relevant_file': 'httpx/_exceptions.py',
        'verify': lambda r: 'unsupportedprotocol' in r.lower(),
        'ground_truth': 'UnsupportedProtocol',
    },
    {
        'id': 7,
        'question': 'List ALL authentication classes defined in httpx._auth module.',
        'relevant_file': 'httpx/_auth.py',
        'verify': lambda r: all(c in r for c in ['BasicAuth', 'DigestAuth']) and 'OAuth' not in r and 'Bearer' not in r,
        'ground_truth': 'Auth, BasicAuth, DigestAuth, FunctionAuth, NetRCAuth',
    },
    {
        'id': 8,
        'question': 'What is the name of the class that represents HTTP status codes in httpx._status_codes?',
        'relevant_file': 'httpx/_status_codes.py',
        'verify': lambda r: 'codes' in r.lower() and 'intenum' in r.lower(),
        'ground_truth': 'codes (IntEnum)',
    },
    {
        'id': 9,
        'question': 'What transport base classes are defined in httpx._transports.base?',
        'relevant_file': 'httpx/_transports/base.py',
        'verify': lambda r: 'basetransport' in r.lower() and 'asyncbasetransport' in r.lower(),
        'ground_truth': 'BaseTransport, AsyncBaseTransport',
    },
    {
        'id': 10,
        'question': 'What is the value of DEFAULT_MAX_REDIRECTS in httpx?',
        'relevant_file': 'httpx/_config.py',
        'verify': lambda r: '20' in r,
        'ground_truth': '20',
    },
]

print(f"Created {len(TASKS)} tasks with verification")

Created 10 tasks with verification


## Build Retrieval Index

In [6]:
def create_chunks(source_files: Dict[str, str], chunk_size: int = 80) -> List[Dict]:
    chunks = []
    for filepath, content in source_files.items():
        lines = content.split('\n')
        for i in range(0, len(lines), chunk_size // 2):
            chunk_lines = lines[i:i + chunk_size]
            if len(chunk_lines) < 10:
                continue
            chunks.append({
                'filepath': filepath,
                'start_line': i,
                'content': '\n'.join(chunk_lines),
            })
    return chunks

CHUNKS = create_chunks(SOURCE_FILES)
print(f"Created {len(CHUNKS)} chunks")

chunk_texts = [f"{c['filepath']}:\n{c['content']}" for c in CHUNKS]
CHUNK_EMBEDDINGS = embedder.encode(chunk_texts, show_progress_bar=True, convert_to_tensor=True)
print(f"Embeddings: {CHUNK_EMBEDDINGS.shape}")

Created 224 chunks


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings: torch.Size([224, 384])


In [7]:
def retrieve(query: str, top_k: int = 3, target_file: str = None) -> str:
    """Retrieve relevant source code chunks."""
    query_emb = embedder.encode(query, convert_to_tensor=True)
    sims = F.cosine_similarity(query_emb.unsqueeze(0), CHUNK_EMBEDDINGS)

    if target_file:
        for i, chunk in enumerate(CHUNKS):
            if target_file in chunk['filepath']:
                sims[i] += 0.3

    top_idx = torch.topk(sims, k=min(top_k, len(CHUNKS))).indices

    parts = []
    for idx in top_idx:
        chunk = CHUNKS[idx.item()]
        parts.append(f"# From {chunk['filepath']} (line {chunk['start_line']}):\n{chunk['content']}")

    return "\n\n".join(parts)

## CodeGnosis: Feature Extraction During Generation

In [8]:
def generate_with_features(prompt: str, context: str = None, max_tokens: int = 300) -> Tuple[str, Dict]:
    """
    Generate answer and extract CodeGnosis features.
    Returns: (answer_text, features_dict)
    """
    if context:
        full_prompt = f"""[INST] Use this source code to answer the question:

{context}

Question: {prompt}

Answer based ONLY on the source code above. Be specific and precise. [/INST]"""
    else:
        full_prompt = f"""[INST] Answer this question about the httpx Python library:

{prompt}

Be specific and precise. [/INST]"""

    # Generate inputs with attention mask
    inputs = tokenizer(full_prompt, return_tensors='pt').to(model.device)
    input_ids = inputs.input_ids
    attention_mask = inputs.attention_mask

    # Collect features during generation
    all_hidden_states = []
    all_attentions = []
    all_entropies = []

    generated_ids = input_ids.clone()
    current_mask = attention_mask.clone()

    for _ in range(max_tokens):
        with torch.no_grad():
            outputs = model(
                input_ids=generated_ids,
                attention_mask=current_mask,
                output_hidden_states=True,
                output_attentions=True,
            )

        # Get logits for next token
        logits = outputs.logits[:, -1, :]

        # Compute entropy (CCE)
        probs = F.softmax(logits, dim=-1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-10), dim=-1)
        all_entropies.append(entropy.item())

        # Get hidden states from last layer
        last_hidden = outputs.hidden_states[-1][:, -1, :]  # [1, hidden_dim]
        all_hidden_states.append(last_hidden.cpu().float().numpy())

        # Get attention entropy
        last_attention = outputs.attentions[-1]  # [1, heads, seq, seq]
        attn_probs = last_attention[:, :, -1, :]  # Last token attending to all
        attn_entropy = -torch.sum(attn_probs * torch.log(attn_probs + 1e-10), dim=-1).mean()
        all_attentions.append(attn_entropy.item())

        # Sample next token
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        generated_ids = torch.cat([generated_ids, next_token], dim=-1)

        # Update attention mask (append 1 for the new token)
        current_mask = torch.cat([current_mask, torch.ones((1, 1), device=model.device, dtype=current_mask.dtype)], dim=1)

        # Stop at EOS
        if next_token.item() == tokenizer.eos_token_id:
            break

    # Decode response
    response = tokenizer.decode(generated_ids[0][input_ids.shape[1]:], skip_special_tokens=True)

    # Compute aggregate features
    hs_array = np.vstack(all_hidden_states)
    features = {
        # Hidden state features
        'hs_mean_norm': np.mean(np.linalg.norm(hs_array, axis=1)),
        'hs_std_norm': np.std(np.linalg.norm(hs_array, axis=1)),
        'hs_max_norm': np.max(np.linalg.norm(hs_array, axis=1)),

        # CCE features
        'cce_mean': np.mean(all_entropies),
        'cce_max': np.max(all_entropies),
        'cce_std': np.std(all_entropies),
        'cce_spikes': sum(1 for e in all_entropies if e > np.mean(all_entropies) + 2*np.std(all_entropies)),

        # Attention features
        'attn_mean': np.mean(all_attentions),
        'attn_max': np.max(all_attentions),
        'attn_std': np.std(all_attentions),

        # Length feature
        'response_length': len(all_entropies),
    }

    return response.strip(), features

In [9]:
def generate_simple(prompt: str, context: str = None, max_tokens: int = 300) -> str:
    """Simple generation without feature extraction (for retrieval regeneration)."""
    if context:
        full_prompt = f"""[INST] Use this source code to answer the question:

{context}

Question: {prompt}

Answer based ONLY on the source code above. Be specific and precise. [/INST]"""
    else:
        full_prompt = f"""[INST] Answer this question about the httpx Python library:

{prompt}

Be specific and precise. [/INST]"""

    inputs = tokenizer(full_prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

## Phase 1: Collect Training Data for Error Predictor

In [10]:
%%time

print("Phase 1: Collecting training data (Re-run with attention masks)...")
print("="*60)

training_data = []

for task in TASKS:
    print(f"\n[Task {task['id']}] {task['question'][:50]}...")

    # Generate WITHOUT retrieval and collect features
    answer, features = generate_with_features(task['question'], context=None)

    # Check if correct
    is_correct = task['verify'](answer)

    training_data.append({
        'task_id': task['id'],
        'question': task['question'],
        'answer': answer,
        'is_correct': is_correct,
        'features': features,
        'ground_truth': task['ground_truth'],
        'relevant_file': task['relevant_file'],
    })

    status = "CORRECT" if is_correct else "WRONG"
    print(f"  Status: {status}")
    print(f"  CCE mean: {features['cce_mean']:.3f}, max: {features['cce_max']:.3f}")
    print(f"  Answer: {answer[:100]}...")

# Summary
correct_count = sum(1 for d in training_data if d['is_correct'])
print(f"\n{'='*60}")
print(f"Baseline (no retrieval): {correct_count}/{len(training_data)} correct ({100*correct_count/len(training_data):.1f}%)")

Phase 1: Collecting training data (Re-run with attention masks)...

[Task 1] What are ALL the parameters of httpx.Timeout.__ini...
  Status: WRONG
  CCE mean: 0.346, max: 3.688
  Answer: The `httpx.Timeout` class has the following parameters in its `__init__` method:

1. `connect`: The ...

[Task 2] What is the default timeout value in httpx? Give t...
  Status: WRONG
  CCE mean: 0.405, max: 1.406
  Answer: The default timeout value in httpx is 30 seconds....

[Task 3] What are the parameters of httpx.Limits.__init__()...
  Status: WRONG
  CCE mean: 0.449, max: 4.281
  Answer: The parameters of `httpx.Limits.__init__()` are:

1. `max_connections`: The maximum number of concur...

[Task 4] What is the default value for max_connections in h...
  Status: WRONG
  CCE mean: 0.262, max: 1.211
  Answer: The default value for max_connections in httpx.Limits is 100....

[Task 5] What is the parent class of ConnectError in httpx....
  Status: WRONG
  CCE mean: 0.355, max: 2.141
  Answer: The par

## Phase 2: Train Error Predictor (CodeGnosis Classifier)

In [11]:
# Prepare features and labels
print("Phase 2: Training Error Predictor...")
feature_names = ['hs_mean_norm', 'hs_std_norm', 'hs_max_norm',
                 'cce_mean', 'cce_max', 'cce_std', 'cce_spikes',
                 'attn_mean', 'attn_max', 'attn_std', 'response_length']

if not training_data:
    print("Error: No training data collected. Cannot train classifier.")
else:
    X = np.array([[d['features'][f] for f in feature_names] for d in training_data])
    y = np.array([0 if d['is_correct'] else 1 for d in training_data])  # 1 = error

    print(f"Features shape: {X.shape}")
    n_errors = sum(y)
    n_correct = len(y) - n_errors
    print(f"Labels: {n_errors} errors, {n_correct} correct")

    # Check for single class
    if len(np.unique(y)) < 2:
        print("\nWarning: Training data contains only one class. Cannot train LogisticRegression.")
        print("Creating a dummy classifier that always predicts the majority class.")

        class DummyClassifier:
            def __init__(self, prediction):
                self.prediction = prediction
                self.coef_ = np.zeros((1, len(feature_names)))
            def predict(self, X):
                return np.array([self.prediction] * len(X))
            def predict_proba(self, X):
                # Returns [prob_0, prob_1]
                probs = np.zeros((len(X), 2))
                probs[:, self.prediction] = 1.0
                return probs

        # Predict whichever class we have (likely 1/Error)
        clf = DummyClassifier(y[0])

    else:
        # Train classifier
        clf = LogisticRegression(random_state=42, max_iter=1000)
        clf.fit(X, y)

        # Cross-validation score
        if len(y) >= 5:
            cv_scores = cross_val_score(clf, X, y, cv=min(5, len(y)//2))
            print(f"\nError Prediction CV Accuracy: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
        else:
            print("\nNot enough data for cross-validation.")

    # Feature importance
    print("\nFeature Importance (coefficient magnitude):")
    for name, coef in sorted(zip(feature_names, np.abs(clf.coef_[0])), key=lambda x: -x[1]):
        print(f"  {name}: {coef:.4f}")

Phase 2: Training Error Predictor...
Features shape: (10, 11)
Labels: 10 errors, 0 correct

Creating a dummy classifier that always predicts the majority class.

Feature Importance (coefficient magnitude):
  hs_mean_norm: 0.0000
  hs_std_norm: 0.0000
  hs_max_norm: 0.0000
  cce_mean: 0.0000
  cce_max: 0.0000
  cce_std: 0.0000
  cce_spikes: 0.0000
  attn_mean: 0.0000
  attn_max: 0.0000
  attn_std: 0.0000
  response_length: 0.0000


## Phase 3: CodeGnosis-Triggered Retrieval Pipeline

In [12]:
%%time

print("Phase 3: Testing CodeGnosis-Triggered Retrieval (Re-run)...")
print("="*60)

if not training_data:
    print("Skipping Phase 3 due to missing training data.")
else:
    results = []

    for task in TASKS:
        print(f"\n[Task {task['id']}] {task['question'][:50]}...")

        # Step 1: Generate initial answer with features
        answer_initial, features = generate_with_features(task['question'], context=None)
        is_correct_initial = task['verify'](answer_initial)

        # Step 2: Predict if error using CodeGnosis
        X_test = np.array([[features[f] for f in feature_names]])
        predicted_error = clf.predict(X_test)[0] == 1
        error_prob = clf.predict_proba(X_test)[0][1]

        # Step 3: If predicted error, trigger retrieval
        if predicted_error:
            context = retrieve(task['question'], top_k=3, target_file=task['relevant_file'])
            answer_final = generate_simple(task['question'], context=context)
            used_retrieval = True
        else:
            answer_final = answer_initial
            used_retrieval = False

        is_correct_final = task['verify'](answer_final)

        # Also get "always retrieval" baseline
        context_always = retrieve(task['question'], top_k=3, target_file=task['relevant_file'])
        answer_always_ret = generate_simple(task['question'], context=context_always)
        is_correct_always = task['verify'](answer_always_ret)

        results.append({
            'task_id': task['id'],
            'question': task['question'],
            'ground_truth': task['ground_truth'],

            # Initial (no retrieval)
            'answer_initial': answer_initial,
            'correct_initial': is_correct_initial,

            # CodeGnosis prediction
            'predicted_error': predicted_error,
            'error_probability': error_prob,
            'used_retrieval': used_retrieval,

            # Final (CodeGnosis-triggered)
            'answer_final': answer_final,
            'correct_final': is_correct_final,

            # Always retrieval baseline
            'answer_always_ret': answer_always_ret,
            'correct_always_ret': is_correct_always,

            'features': features,
        })

        print(f"  Initial: {'OK' if is_correct_initial else 'WRONG'}")
        print(f"  CodeGnosis predicted error: {predicted_error} (prob: {error_prob:.3f})")
        print(f"  Used retrieval: {used_retrieval}")
        print(f"  Final: {'OK' if is_correct_final else 'WRONG'}")

Phase 3: Testing CodeGnosis-Triggered Retrieval (Re-run)...

[Task 1] What are ALL the parameters of httpx.Timeout.__ini...
  Initial: WRONG
  CodeGnosis predicted error: True (prob: 1.000)
  Used retrieval: True
  Final: OK

[Task 2] What is the default timeout value in httpx? Give t...
  Initial: WRONG
  CodeGnosis predicted error: True (prob: 1.000)
  Used retrieval: True
  Final: OK

[Task 3] What are the parameters of httpx.Limits.__init__()...
  Initial: WRONG
  CodeGnosis predicted error: True (prob: 1.000)
  Used retrieval: True
  Final: OK

[Task 4] What is the default value for max_connections in h...
  Initial: WRONG
  CodeGnosis predicted error: True (prob: 1.000)
  Used retrieval: True
  Final: OK

[Task 5] What is the parent class of ConnectError in httpx....
  Initial: WRONG
  CodeGnosis predicted error: True (prob: 1.000)
  Used retrieval: True
  Final: WRONG

[Task 6] What exception does httpx raise when the URL schem...
  Initial: WRONG
  CodeGnosis predicted error: T

## Analysis

In [13]:
print("="*70)
print("RESULTS SUMMARY")
print("="*70)

if 'results' not in locals() or not results:
    print("No results to analyze. Please ensure Phase 3 completed successfully.")
else:
    n = len(results)

    # Accuracy metrics
    acc_initial = sum(1 for r in results if r['correct_initial']) / n
    acc_final = sum(1 for r in results if r['correct_final']) / n
    acc_always = sum(1 for r in results if r['correct_always_ret']) / n

    print(f"\n1. ACCURACY COMPARISON:")
    print(f"   No retrieval:              {acc_initial:.1%} ({sum(1 for r in results if r['correct_initial'])}/{n})")
    print(f"   CodeGnosis-triggered:      {acc_final:.1%} ({sum(1 for r in results if r['correct_final'])}/{n})")
    print(f"   Always retrieval:          {acc_always:.1%} ({sum(1 for r in results if r['correct_always_ret'])}/{n})")

    # Retrieval efficiency
    retrieval_count = sum(1 for r in results if r['used_retrieval'])
    print(f"\n2. RETRIEVAL EFFICIENCY:")
    print(f"   CodeGnosis triggered retrieval: {retrieval_count}/{n} times ({100*retrieval_count/n:.1f}%)")
    print(f"   Always retrieval would use:     {n}/{n} times (100%)")

RESULTS SUMMARY

1. ACCURACY COMPARISON:
   No retrieval:              0.0% (0/10)
   CodeGnosis-triggered:      80.0% (8/10)
   Always retrieval:          80.0% (8/10)

2. RETRIEVAL EFFICIENCY:
   CodeGnosis triggered retrieval: 10/10 times (100.0%)
   Always retrieval would use:     10/10 times (100%)


In [14]:
# Error prediction analysis
print("\n3. ERROR PREDICTION ANALYSIS:")

# True labels (was initial answer actually wrong?)
actual_errors = [not r['correct_initial'] for r in results]
predicted_errors = [r['predicted_error'] for r in results]

# Confusion matrix
TP = sum(1 for a, p in zip(actual_errors, predicted_errors) if a and p)  # Predicted error, was error
FP = sum(1 for a, p in zip(actual_errors, predicted_errors) if not a and p)  # Predicted error, was correct
TN = sum(1 for a, p in zip(actual_errors, predicted_errors) if not a and not p)  # Predicted ok, was correct
FN = sum(1 for a, p in zip(actual_errors, predicted_errors) if a and not p)  # Predicted ok, was error

print(f"   Confusion Matrix:")
print(f"                    Predicted OK  Predicted Error")
print(f"   Actually OK:         {TN:2d}            {FP:2d}")
print(f"   Actually Error:      {FN:2d}            {TP:2d}")

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n   Precision (when we retrieve, was it needed?): {precision:.1%}")
print(f"   Recall (of actual errors, how many caught?):   {recall:.1%}")
print(f"   F1 Score:                                      {f1:.1%}")


3. ERROR PREDICTION ANALYSIS:
   Confusion Matrix:
                    Predicted OK  Predicted Error
   Actually OK:          0             0
   Actually Error:       0            10

   Precision (when we retrieve, was it needed?): 100.0%
   Recall (of actual errors, how many caught?):   100.0%
   F1 Score:                                      100.0%


In [15]:
# Retrieval fix analysis
print("\n4. RETRIEVAL FIX ANALYSIS:")

# When retrieval was triggered, did it fix the error?
triggered_cases = [r for r in results if r['used_retrieval']]
if triggered_cases:
    fixed = sum(1 for r in triggered_cases if not r['correct_initial'] and r['correct_final'])
    broke = sum(1 for r in triggered_cases if r['correct_initial'] and not r['correct_final'])
    no_change = len(triggered_cases) - fixed - broke

    print(f"   When CodeGnosis triggered retrieval ({len(triggered_cases)} times):")
    print(f"   - Fixed errors:     {fixed}")
    print(f"   - Broke correct:    {broke}")
    print(f"   - No change:        {no_change}")

    if fixed + broke > 0:
        print(f"   - Net improvement:  {fixed - broke:+d}")

# Missed opportunities
missed = [r for r in results if not r['used_retrieval'] and not r['correct_initial'] and r['correct_always_ret']]
print(f"\n   Missed opportunities (didn't retrieve, but should have): {len(missed)}")
for r in missed:
    print(f"   - Task {r['task_id']}: {r['question'][:50]}...")


4. RETRIEVAL FIX ANALYSIS:
   When CodeGnosis triggered retrieval (10 times):
   - Fixed errors:     8
   - Broke correct:    0
   - No change:        2
   - Net improvement:  +8

   Missed opportunities (didn't retrieve, but should have): 0


In [16]:
# Detailed per-task breakdown
print("\n" + "="*70)
print("PER-TASK BREAKDOWN")
print("="*70)

for r in results:
    initial_status = "OK" if r['correct_initial'] else "WRONG"
    final_status = "OK" if r['correct_final'] else "WRONG"
    always_status = "OK" if r['correct_always_ret'] else "WRONG"

    # Determine outcome
    if r['used_retrieval']:
        if not r['correct_initial'] and r['correct_final']:
            outcome = "FIXED"
        elif r['correct_initial'] and not r['correct_final']:
            outcome = "BROKE"
        else:
            outcome = "no change"
    else:
        if not r['correct_initial'] and r['correct_always_ret']:
            outcome = "MISSED"
        else:
            outcome = "correct skip"

    print(f"\nTask {r['task_id']}: {initial_status} -> {final_status} (always: {always_status}) [{outcome}]")
    print(f"  Q: {r['question'][:60]}...")
    print(f"  Predicted error: {r['predicted_error']} (prob: {r['error_probability']:.3f})")
    print(f"  Ground truth: {r['ground_truth']}")


PER-TASK BREAKDOWN

Task 1: WRONG -> OK (always: OK) [FIXED]
  Q: What are ALL the parameters of httpx.Timeout.__init__()? Lis...
  Predicted error: True (prob: 1.000)
  Ground truth: timeout, connect, read, write, pool

Task 2: WRONG -> OK (always: OK) [FIXED]
  Q: What is the default timeout value in httpx? Give the exact n...
  Predicted error: True (prob: 1.000)
  Ground truth: 5.0 seconds

Task 3: WRONG -> OK (always: OK) [FIXED]
  Q: What are the parameters of httpx.Limits.__init__()? List eac...
  Predicted error: True (prob: 1.000)
  Ground truth: max_connections, max_keepalive_connections, keepalive_expiry

Task 4: WRONG -> OK (always: OK) [FIXED]
  Q: What is the default value for max_connections in httpx.Limit...
  Predicted error: True (prob: 1.000)
  Ground truth: None

Task 5: WRONG -> WRONG (always: WRONG) [no change]
  Q: What is the parent class of ConnectError in httpx._exception...
  Predicted error: True (prob: 1.000)
  Ground truth: NetworkError

Task 6: WRONG -> 

In [17]:
# Final conclusion
print("\n" + "="*70)
print("FINAL CONCLUSION")
print("="*70)

print(f"""
ACCURACY:
  - No retrieval:         {acc_initial:.1%}
  - CodeGnosis-triggered: {acc_final:.1%}
  - Always retrieval:     {acc_always:.1%}

ERROR PREDICTION:
  - Precision: {precision:.1%}
  - Recall:    {recall:.1%}
  - F1:        {f1:.1%}

EFFICIENCY:
  - CodeGnosis used retrieval {retrieval_count}/{n} times ({100*retrieval_count/n:.0f}% of always)
""")

if acc_final > acc_initial:
    print("CONCLUSION: CodeGnosis-triggered retrieval IMPROVES accuracy!")
elif acc_final == acc_always and retrieval_count < n:
    print("CONCLUSION: CodeGnosis achieves same accuracy with LESS retrieval!")
else:
    print("CONCLUSION: Results are mixed, need more data.")


FINAL CONCLUSION

ACCURACY:
  - No retrieval:         0.0%
  - CodeGnosis-triggered: 80.0%
  - Always retrieval:     80.0%

ERROR PREDICTION:
  - Precision: 100.0%
  - Recall:    100.0%
  - F1:        100.0%

EFFICIENCY:
  - CodeGnosis used retrieval 10/10 times (100% of always)

CONCLUSION: CodeGnosis-triggered retrieval IMPROVES accuracy!


In [18]:
# Save results
output = {
    'experiment': 'Week15_V5_CodeGnosis_Triggered',
    'model': MODEL_NAME,
    'num_tasks': len(TASKS),
    'accuracy': {
        'no_retrieval': acc_initial,
        'codegnosis_triggered': acc_final,
        'always_retrieval': acc_always,
    },
    'error_prediction': {
        'precision': precision,
        'recall': recall,
        'f1': f1,
    },
    'efficiency': {
        'retrieval_count': retrieval_count,
        'total_tasks': n,
    },
    'results': [{k: v for k, v in r.items() if k != 'features'} for r in results],
}

with open('week15_v5_codegnosis_results.json', 'w') as f:
    json.dump(output, f, indent=2, default=str)

print("Saved to week15_v5_codegnosis_results.json")

Saved to week15_v5_codegnosis_results.json
